### RAG Pipelines- Data Ingestion to Vector DB

In [69]:
import os
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [70]:
def process_all_pdfs(directory):
    all_docs = []
    pdf_dir = Path(directory)
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(f"Found {len(pdf_files)} Pdf file to process")

    for pdf_file in pdf_files:
        print(f"\n processing :{pdf_file.name}")
        try:
            loader = PyMuPDFLoader(str(pdf_file))
            documents = loader.load()
            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = "pdf"


            all_docs.extend(documents)
            print(f"Loaded {len(documents)} pages")
        except Exception as e:
            print(f"Error {e}")
    print(f"\n Total docs loaded : {len(all_docs)}")
    return all_docs

all_pdfs = process_all_pdfs("../data")

Found 2 Pdf file to process

 processing :embeddings_learning_guide.pdf
Loaded 7 pages

 processing :object_detection_learning_guide.pdf
Loaded 14 pages

 Total docs loaded : 21


In [71]:
# def pdf_loader():
#     pdf_loader = DirectoryLoader(
#         "../data/pdf",
#         glob = "**/*.pdf",
#         loader_cls=PyMuPDFLoader,
#     )
#     docs = pdf_loader.load()
#     all_docs =[]
#     for doc in docs:
#         # print(docs)
#         source_path = Path(doc.metadata['source'])

#         # print(source_path.name)
#         doc.metadata['source_file'] = source_path.name
#         doc.metadata['file_type'] = 'pdf'
#         all_docs.extend(doc)
#     return all_docs
# all_pdf_docs = pdf_loader()

In [72]:
all_pdfs

[Document(metadata={'producer': 'LibreOffice 25.2.3.2 (X86_64) / LibreOffice Community', 'creator': 'Writer', 'creationdate': '2026-07-14T13:46:52+00:00', 'source': '..\\data\\pdf\\embeddings_learning_guide.pdf', 'file_path': '..\\data\\pdf\\embeddings_learning_guide.pdf', 'total_pages': 7, 'format': 'PDF 1.7', 'title': '', 'author': 'python-docx', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': "D:20260714134652Z'", 'page': 0, 'source_file': 'embeddings_learning_guide.pdf', 'file_type': 'pdf'}, page_content='LLM Foundations - Embeddings\nEmbedding Learning Guide\nBEGINNER-FRIENDLY LEARNING NOTE\nEmbeddings\nFrom text to meaningful numerical vectors\nEnglish definitions + Hinglish explanations + RAG examples\nLearning goal\nBy the end of this guide, you will understand what an embedding is, why it is required, how semantic \nsimilarity works, and how embeddings are used inside a RAG pipeline.\nPrepared for: Leelamaya | Date: 14 July 2026'),
 

In [73]:
## tEXT SPLITTING GET INTO CHUNKS
# for me i have to test can i increase or decrease the chunk var is there any fixed size of cjhunking
def split_documents(documents, chunk_size = 1000, chunk_overlap = 200):
    text_splitter  = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators = ["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    if split_docs:
        print(f"\n Example chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [74]:
chunks = split_documents(all_pdfs)
chunks

Split 21 documents into 42 chunks

 Example chunk:
Content: LLM Foundations - Embeddings
Embedding Learning Guide
BEGINNER-FRIENDLY LEARNING NOTE
Embeddings
From text to meaningful numerical vectors
English definitions + Hinglish explanations + RAG examples
Le...
Metadata: {'producer': 'LibreOffice 25.2.3.2 (X86_64) / LibreOffice Community', 'creator': 'Writer', 'creationdate': '2026-07-14T13:46:52+00:00', 'source': '..\\data\\pdf\\embeddings_learning_guide.pdf', 'file_path': '..\\data\\pdf\\embeddings_learning_guide.pdf', 'total_pages': 7, 'format': 'PDF 1.7', 'title': '', 'author': 'python-docx', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': "D:20260714134652Z'", 'page': 0, 'source_file': 'embeddings_learning_guide.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'LibreOffice 25.2.3.2 (X86_64) / LibreOffice Community', 'creator': 'Writer', 'creationdate': '2026-07-14T13:46:52+00:00', 'source': '..\\data\\pdf\\embeddings_learning_guide.pdf', 'file_path': '..\\data\\pdf\\embeddings_learning_guide.pdf', 'total_pages': 7, 'format': 'PDF 1.7', 'title': '', 'author': 'python-docx', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': "D:20260714134652Z'", 'page': 0, 'source_file': 'embeddings_learning_guide.pdf', 'file_type': 'pdf'}, page_content='LLM Foundations - Embeddings\nEmbedding Learning Guide\nBEGINNER-FRIENDLY LEARNING NOTE\nEmbeddings\nFrom text to meaningful numerical vectors\nEnglish definitions + Hinglish explanations + RAG examples\nLearning goal\nBy the end of this guide, you will understand what an embedding is, why it is required, how semantic \nsimilarity works, and how embeddings are used inside a RAG pipeline.\nPrepared for: Leelamaya | Date: 14 July 2026'),
 

### Embeding and vector DB

In [75]:
### 
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List,Dict,Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [76]:
class EmbeddingManager:
    """ Handels document embedding generation using SentenceTransformer"""
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager

        Args: Huggingface modelname for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the sentence transformer Model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"model loaded successfully. Embeddng dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise


    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate Embedding for a list of texts
        Args: List of text strings to embed

        Returns: numpy array of embeddings with shape(len(texts), embedding_dim)
        """

        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embedding for {len(texts)} texts ...")
        embeddings = self.model.encode(texts, show_progress_bar= True)
        print(f"Generated embedding with shape: {embeddings.shape}")
        return embeddings
    def get_embedding_dimension(self) -> int:
        """ Get the embedding dimension of the model"""
        if not self.model:
            raise ValueError("Model not loaded")
        return self.model.get_sentence_embedding_dimension()
    
## Initialize the embedding manager

embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8591.13it/s]


model loaded successfully. Embeddng dimension: 384


C:\Users\PC\AppData\Local\Temp\ipykernel_5768\1335580026.py:18: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"model loaded successfully. Embeddng dimension: {self.model.get_sentence_embedding_dimension()}")


### vector Store

In [77]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store."""

    def __init__(
        self,
        collection_name: str = "pdf_documents",
        persist_directory: str = "../data/vector_store",
    ):
        """
        Initialize the vector store.

        Args:
            collection_name: Name of the ChromaDB collection.
            persist_directory: Directory used to persist the vector store.
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None

        self._initialize_store()

    def _initialize_store(self):
        """Initialize the ChromaDB client and collection."""

        try:
            # Create the directory if it does not exist
            os.makedirs(self.persist_directory, exist_ok=True)

            # Create persistent ChromaDB client
            self.client = chromadb.PersistentClient(
                path=self.persist_directory
            )

            # Get an existing collection or create a new one
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={
                    "description": "PDF document embeddings for RAG"
                },
            )

            print(
                f"Vector store initialized. "
                f"Collection: {self.collection_name}"
            )

            print(
                f"Existing documents in collection: "
                f"{self.collection.count()}"
            )

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(
        self,
        documents: List[Any],
        embeddings: np.ndarray,
    ):
        """
        Add documents and their embeddings to the vector store.

        Args:
            documents: List of LangChain Document objects.
            embeddings: Embeddings corresponding to the documents.
        """

        if len(documents) != len(embeddings):
            raise ValueError(
                "Number of documents must match number of embeddings."
            )

        if len(documents) == 0:
            print("No documents available to add.")
            return

        print(
            f"Adding {len(documents)} documents to vector store..."
        )

        ids = []
        metadatas = []
        document_texts = []
        embeddings_list = []

        for index, (doc, embedding) in enumerate(
            zip(documents, embeddings)
        ):
            # Generate unique document ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{index}"
            ids.append(doc_id)

            # Copy existing LangChain metadata
            metadata = dict(doc.metadata)

            # Add custom metadata
            metadata["doc_index"] = index
            metadata["content_length"] = len(doc.page_content)

            metadatas.append(metadata)

            # Store document content
            document_texts.append(doc.page_content)

            # Convert NumPy embedding to normal Python list
            if isinstance(embedding, np.ndarray):
                embeddings_list.append(embedding.tolist())
            else:
                embeddings_list.append(list(embedding))

        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=document_texts,
            )

            print(
                f"Successfully added {len(documents)} documents "
                f"to vector store."
            )

            print(
                f"Total documents in collection: "
                f"{self.collection.count()}"
            )

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise


vectorstore = VectorStore()

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 42


In [78]:
chunks

[Document(metadata={'producer': 'LibreOffice 25.2.3.2 (X86_64) / LibreOffice Community', 'creator': 'Writer', 'creationdate': '2026-07-14T13:46:52+00:00', 'source': '..\\data\\pdf\\embeddings_learning_guide.pdf', 'file_path': '..\\data\\pdf\\embeddings_learning_guide.pdf', 'total_pages': 7, 'format': 'PDF 1.7', 'title': '', 'author': 'python-docx', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': "D:20260714134652Z'", 'page': 0, 'source_file': 'embeddings_learning_guide.pdf', 'file_type': 'pdf'}, page_content='LLM Foundations - Embeddings\nEmbedding Learning Guide\nBEGINNER-FRIENDLY LEARNING NOTE\nEmbeddings\nFrom text to meaningful numerical vectors\nEnglish definitions + Hinglish explanations + RAG examples\nLearning goal\nBy the end of this guide, you will understand what an embedding is, why it is required, how semantic \nsimilarity works, and how embeddings are used inside a RAG pipeline.\nPrepared for: Leelamaya | Date: 14 July 2026'),
 

In [79]:
### Convert text to embeddngs

texts = [doc.page_content for doc in chunks]




### generate the embedding

embeddings = embedding_manager.generate_embeddings(texts)

#### Store inside Vector database

vectorstore.add_documents(chunks,embeddings)

Generating embedding for 42 texts ...


Batches: 100%|██████████| 2/2 [00:00<00:00,  3.46it/s]

Generated embedding with shape: (42, 384)
Adding 42 documents to vector store...
Successfully added 42 documents to vector store.
Total documents in collection: 84


### Retrival Pipeline---- RAG Retriver

In [88]:
class RAGRetriver:
    """ Handels query_based retrival from the vector store """
    def __init__(self, vector_store : VectorStore, embedding_manager: EmbeddingManager):
        """Initialize the retriver
        Args:
        vector_store: Vector store containing document embeddings
        embedding_manager: manager for generating query embeddings
        """
        self. vector_store = vector_store
        self. embedding_manager = embedding_manager
    
    def retrive(self, query: str, top_k: int = 5, score_threshold:float=0.0) -> List[Dict[str,Any]]:
        """Retrive Relevant documents fora query
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
            Returns:
                List of dictionaries containing retrived documents and metadata
        """

        print(f"Retriving documents for qurery: '{query}'")
        print(f"Top-K: {top_k}, score_threshold: {score_threshold}")

        #generate query embedding
        query_embedding = self. embedding_manager.generate_embeddings([query])[0]

        # Search in vector store

        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results = top_k
            )

            # Process Results
            retrived_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents,metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)

                    similarity_score = 1- distance

                    if similarity_score >= score_threshold:
                        retrived_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i+1
                        })

                        print(f"Retrived {len(retrived_docs)} documents (after filtering)")

                    else:
                        print("No documents found")
                return retrived_docs
            
        except Exception as e:
            print(f"Error during retrival: {e}")
            return []



rag_retriver = RAGRetriver(vectorstore, embedding_manager)



In [89]:
rag_retriver

In [82]:
rag_retriver.retrive("What is object detection")

Retriving documents for qurery: 'What is object detection'
Top-K: 5, score_threshold: 0.0
Generating embedding for 1 texts ...


Batches: 100%|██████████| 1/1 [00:00<00:00, 178.09it/s]

Generated embedding with shape: (1, 384)
Retrived 1 documents (after filtering)
Retrived 2 documents (after filtering)
Retrived 3 documents (after filtering)
Retrived 4 documents (after filtering)
Retrived 5 documents (after filtering)


[{'id': 'doc_7b6b2304_17',
  'content': 'Object Detection Learning Guide  |  Page 3\n1. What Is Object Detection?\nObject detection is a computer vision task that identifies which objects are present in an image or video and \nlocates each object using a bounding box. A detector normally returns a class label, box coordinates, and a \nconfidence score for every detected object.\nHinglish explanation\nObject detection ka matlab sirf image me kya hai batana nahi hota. Model ko yeh bhi batana hota hai ki object \nimage ke kis area me hai. Isliye output me class name ke saath bounding box aur confidence score milta hai.\nWhy detection is harder than classification\n\uf0b7\nAn image may contain zero, one, or many objects.\n\uf0b7\nObjects can appear at different sizes, angles, lighting conditions, and distances.\n\uf0b7\nThe model must solve two problems together: classification and localization.\n\uf0b7\nOverlapping objects, occlusion, blur, and complex backgrounds increase difficulty.\nCo

In [90]:
rag_retriver.retrive("What is an Embedding?")

Retriving documents for qurery: 'What is an Embedding?'
Top-K: 5, score_threshold: 0.0
Generating embedding for 1 texts ...


Batches: 100%|██████████| 1/1 [00:00<00:00, 137.24it/s]

Generated embedding with shape: (1, 384)
Retrived 1 documents (after filtering)
Retrived 2 documents (after filtering)
Retrived 3 documents (after filtering)
Retrived 4 documents (after filtering)
Retrived 5 documents (after filtering)


[{'id': 'doc_7d0e636f_0',
  'content': 'LLM Foundations - Embeddings\nEmbedding Learning Guide\nBEGINNER-FRIENDLY LEARNING NOTE\nEmbeddings\nFrom text to meaningful numerical vectors\nEnglish definitions + Hinglish explanations + RAG examples\nLearning goal\nBy the end of this guide, you will understand what an embedding is, why it is required, how semantic \nsimilarity works, and how embeddings are used inside a RAG pipeline.\nPrepared for: Leelamaya | Date: 14 July 2026',
  'metadata': {'producer': 'LibreOffice 25.2.3.2 (X86_64) / LibreOffice Community',
   'source': '..\\data\\pdf\\embeddings_learning_guide.pdf',
   'modDate': '',
   'creationdate': '2026-07-14T13:46:52+00:00',
   'moddate': '',
   'file_path': '..\\data\\pdf\\embeddings_learning_guide.pdf',
   'total_pages': 7,
   'doc_index': 0,
   'creator': 'Writer',
   'title': '',
   'file_type': 'pdf',
   'keywords': '',
   'page': 0,
   'source_file': 'embeddings_learning_guide.pdf',
   'trapped': '',
   'subject': '',
   'c

### Integration Vector db context pipeline with LLm Output

In [91]:
### simple RAG Pipeline with graq LLM

from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

#### Initialize the groq llm
# os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
groq_api_key = os.getenv("GROQ_API_KEY")

llm  = ChatGroq(groq_api_key= groq_api_key, model_name = "qwen/qwen3-32b", temperature=0.1, max_tokens=1024)

## 2. simple RAG ffunction : retrive_context + generate response

def rag_qa(query, retriver, llm, top_k=3):
    ## retriver the context

    results = retriver.retrive(query,top_k=top_k)
    context = "\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context :
        return "No Relevant context found to answer the question"
    
    ## generate the answer using groq llm

    prompt = f"""Use the following context to answer the question consisely.
    Context: {context}
    Question:{query}
    Answer:"""


    response = llm.invoke([prompt.format(context = context, query=query)])
    return response.content


In [85]:
answer = rag_qa("What is embeddings?", rag_retriver,llm)
print(answer)

Retriving documents for qurery: 'What is embeddings?'
Top-K: 3, score_threshold: 0.0
Generating embedding for 1 texts ...


Batches: 100%|██████████| 1/1 [00:00<00:00, 100.34it/s]

Generated embedding with shape: (1, 384)
Retrived 1 documents (after filtering)
Retrived 2 documents (after filtering)
Retrived 3 documents (after filtering)


<think>
Okay, the user is asking, "What is embeddings?" Let me start by recalling the context provided. The context is about LLM Foundations, specifically embeddings. The key points mentioned are that embeddings are vectors in a semantic space where similar items are closer. They are dense vectors, meaning most elements are non-zero, and they're used for storage and comparison via similarity calculations.

First, I need to define embeddings in simple terms. The user might be a beginner, so I should avoid jargon. The context mentions that embeddings convert text into numerical vectors. I should explain that these vectors capture meaning based on context. Also, the important point is that individual numbers aren't interpreted; instead, the whole vector's relationships matter.

I should mention the semantic vector space part—how similar items are closer. Maybe use an example like "king" and "queen" being near each other. Also, note that real embeddings have hundreds or thousands of dimens

In [93]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriver, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriver.retrive(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output
result = rag_advanced(
    "What is object detection?",
    rag_retriver,
    llm,
    top_k=3,
    min_score=0.1,
    return_context=True
)

print("Answer:", result['answer'])
print("Sources:", result['sources'])

print("Answer:", result['confidence'])
print("Answer:", result['context'][:300])


Retriving documents for qurery: 'What is object detection?'
Top-K: 3, score_threshold: 0.1
Generating embedding for 1 texts ...


Batches: 100%|██████████| 1/1 [00:00<00:00, 111.15it/s]

Generated embedding with shape: (1, 384)
Retrived 1 documents (after filtering)
Retrived 2 documents (after filtering)
Retrived 3 documents (after filtering)


Answer: <think>
Okay, let's see. The user is asking for a concise answer to "What is object detection?" based on the provided context. The context is from a learning guide, specifically page 3. 

First, I need to recall the key points from the context. The first section explains that object detection is a computer vision task that identifies objects in images or videos and locates them with bounding boxes. The detector returns class labels, box coordinates, and confidence scores. There's also a Hinglish explanation emphasizing that it's not just about identifying what's in the image but also where, hence the bounding box and confidence score.

The user wants the answer to be concise. So I should start by stating the definition clearly. Mention that it's a computer vision task. Then note the two main aspects: identifying objects and locating them with bounding boxes. Also, include the three outputs: class label, box coordinates, and confidence score. 

I need to make sure not to include

In [94]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriver, llm)
result = adv_rag.query("what is attention is all you need", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

AttributeError: 'RAGRetriver' object has no attribute 'retrieve'